# Welcome!
This notebook demonstrates how to develop a conversational system that uses a deep knowledge base about hotels, combining structured instance-level data and an ontological model. The knowledge graph (KG) and ontology enable reasoning to enhance dialogue response generation. The task involves integrating a GraphRAG-like approach to query the knowledge base and generate accurate, context-aware responses.

Specifically, the notebook has the following steps:

1. **Setup**: Loading the knowledge graph, dialogues, and required libraries (e.g., OWLAPY).
2. **Analyzing the knowledge graph**: Exploring its structure and entities using OWLAPY queries.
3. **Extending the ontology**: Adding TBox information for expressive reasoning.
4. **Creating dialogues**: Create dialogues based on the examples. Write 5 simple dialogues and 5 more detailed ones to showcase different types of interactions.
5. **Combining ontology and KG data**: Deploying an OWL reasoner to perform class-expression queries.
6. **Query generation with LLMs**: Using an LLM (e.g., Llama3.2) to generate or assist in creating queries against the KG.
7. **Generating responses**: Summarizing retrieved data into dialogue responses using a KG-augmented RAG approach.
8. **Evaluation**: Assessing the system's performance using metrics like intersection-over-union scores.

## Assignment
The goal of this assignment is to develop a logic-enhanced conversational system that retrieves and reasons over domain knowledge to assist in dialogue response generation. You will focus on both the technical aspects of KG+ontology reasoning and the integration with LLMs for robust responses.

### Assignment Steps
1. **Analyze the provided knowledge graph and dialogues**:
   - Explore the KG's entities, properties, and relevance to the dialogues.
   - Identify opportunities where ontology reasoning enhances dialogue responses.
2. **Extend the ontology**:
   - Add expressive TBox information to support meaningful inferences.
3. **Deploy the reasoning environment**:
   - Use OWLAPY to combine the KG (as ABox) with the ontology for reasoning-based queries.
4. **Generate class-expression queries**:
   - Use instruction-based, few-shot prompting with Llama3.2 to produce or assist in creating the queries.
5. **Summarize results into dialogue responses**:
   - Apply KG-augmented RAG to generate user-facing answers based on reasoning results.
6. **Evaluate the system**:
   - Use appropriate metrics, including intersection-over-union scores for set-based answers.

## Report
Write a **5-page report** in LNCS format that includes:

1. **Introduction**: Background on conversational systems with LLMs and the role of reasoning over domain knowledge.
2. **Methodology**: A detailed description of your approach, including diagrams and examples.
3. **Results**: Evaluation findings from the implemented steps.
4. **Discussion**: Strengths and weaknesses of your approach, lessons learned, and potential improvements.

Make sure to use the following template: [Springer Lecture Notes in Computer Science](https://www.overleaf.com/latex/templates/springer-lecture-notes-in-computer-science/kzwwpvhwnvfj)


## Grading
Your work will be evaluated based on:

1. **Code Implementation (30%)**: Quality and functionality of the logic-enhanced conversational system.
2. **Report (70%)**: Depth of analysis and clarity in presenting methods, results, and lessons learned.

## Kaggle Environment Notes
To ensure smooth execution:
- Load the required data into `/kaggle/input/`.
- Use `/kaggle/working/` for saving temporary files.
- Turn on GPUs and internet connectivity when necessary, and follow best practices for resource management.

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input director

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/w1-data/examples.txt
/kaggle/input/w1-data/data.ttl


# Install packages

In [3]:
!pip install jpype1==1.5.2
!pip install owlapy==1.5.1
!pip install ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 493.5/493.5 kB 9.4 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 77.4 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 74.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.2/587.2 kB 34.0 MB/s eta 0:00:00
  Created wheel for owlready2: filename=owlready2-0.49-py3-none-any.whl size=23742426 sha256=d9ea23979fe56df71bd4b6da05b29a5e12ba752256032f896471c79da5edf795
  Stored in directory: /root/.cache/pip/wheels/43/fe/dc/a0de3c289cfd5923ece6524469d328950e14fa0c90b1088ffa
Successfully built owlready2


# Import libraries


In [4]:
from owlapy import manchester_to_owl_expression, dl_to_owl_expression
from owlapy.iri import IRI
from owlapy.owl_ontology import Ontology
from owlapy.owl_reasoner import SyncReasoner, StructuralReasoner

# 1. Analyze the provided knowledge graph (data.ttl).

In [20]:
## the provided knowledge graph is in turtle (.ttl) format, which owlready2 has trouble
## parsing correctly in this environment. to avoid this issue, we first load the file
## using rdflib (which fully supports turtle), convert it to n-triples,
## and then load the converted graph into owlready2 for analysis.
## for the record owlready2 is an inner library used by owlapy.

from rdflib import Graph
from owlready2 import World

src = "/kaggle/input/w1-data2/extended_data.ttl"
dst = "/kaggle/working/data.nt"   ## .nt is the file format for n-triples

g = Graph()

g.parse(src, format="turtle")   ## here we parse the .ttl
g.serialize(destination=dst, format="nt")   # here its converted into .nt

world = World()
onto = world.get_ontology(f"file://{dst}").load(format="ntriples")

print("loaded into:", onto.base_iri)

loaded into: file:///kaggle/working/data.nt#


/usr/local/lib/python3.12/dist-packages/rdflib/plugins/serializers/nt.py:39: UserWarning: NTSerializer always uses UTF-8 encoding. Given encoding was: None
  warnings.warn(


In [21]:
from collections import Counter
from rdflib import URIRef, Literal
import pandas as pd

# helper funcs
def is_uri(x): 
    return isinstance(x, URIRef)

def is_lit(x):
    return isinstance(x, Literal)

def shorten(term, graph):
    # compact display using namespaces when possible
    
    if isinstance(term, URIRef):
        try:
            return term.n3(graph.namespace_manager)
        except Exception:
            return str(term)
    if isinstance(term, Literal):
        if term.language:
            return f"\"{str(term)[:60]}\"@{term.language}"
        if term.datatype:
            return f"\"{str(term)[:60]}\"^^{term.datatype}"
        return f"\"{str(term)[:60]}\""
    return str(term)


triples = list(g.triples((None, None, None)))
print("--- basic info ---")
print("triples:", len(triples))

subjects = set(s for s, p, o in triples)
predicates = set(p for s, p, o in triples)
objects = set(o for s, p, o in triples)

uris = set(x for x in subjects.union(objects) if is_uri(x))
lits = set(x for x in objects if is_lit(x))

print("unique subjects:", len(subjects))
print("unique predicates:", len(predicates))
print("unique objects:", len(objects))
print("unique URI nodes (subjects U objects):", len(uris))
print("unique literal nodes (objects):", len(lits))

pred_counts = Counter(p for s, p, o in triples)
top_preds = pred_counts.most_common(30)

print()
print("--- top predicates (by triple count) ---")

df_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "count"]
)
display(df_preds.head(30))

RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

type_triples = list(g.triples((None, RDF_TYPE, None)))
class_counts = Counter(o for s, p, o in type_triples if is_uri(o))

print()
print("--- types / classes (rdf:type) ---")
print("rdf:type triples:", len(type_triples))
print("distinct classes:", len(class_counts))

df_classes = pd.DataFrame(
    [(str(cls), shorten(cls, g), c) for cls, c in class_counts.most_common()],
    columns=["class_iri", "class", "instances_count"]
)
display(df_classes.head(30))

datatype_counts = Counter()
lang_counts = Counter()
lit_pred_counts = Counter()
lit_lengths = []

for s, p, o in triples:
    if is_lit(o):
        lit_pred_counts[p] += 1
        if o.datatype:
            datatype_counts[o.datatype] += 1
        else:
            datatype_counts[None] += 1
        if o.language:
            lang_counts[o.language] += 1
        lit_lengths.append(len(str(o)))

print()
print("--- literals ---")
print("literal objects:", sum(lit_pred_counts.values()))
print("predicates with literals:", len(lit_pred_counts))
print("avg literal length:", (sum(lit_lengths) / len(lit_lengths)) if lit_lengths else 0)

print("top literal predicates:")
for p, c in lit_pred_counts.most_common(20):
    print(f"{c:>7}  {shorten(p, g)}")

print("top datatypes:")
for dt, c in datatype_counts.most_common(15):
    dt_name = "no-datatype" if dt is None else shorten(dt, g)
    print(f"{c:>7}  {dt_name}")

print("top languages:")
for lang, c in lang_counts.most_common(10):
    print(f"{c:>7}  {lang}")

df_lit_preds = pd.DataFrame(
    [(str(p), shorten(p, g), c) for p, c in lit_pred_counts.most_common()],
    columns=["predicate_iri", "predicate", "literal_count"]
)
display(df_lit_preds.head(30))

--- basic info ---
triples: 9228
unique subjects: 1789
unique predicates: 12
unique objects: 870
unique URI nodes (subjects U objects): 1792
unique literal nodes (objects): 476

--- top predicates (by triple count) ---


,predicate_iri,predicate,count
0,http://www.w3.org/1999/02/22-rdf-syntax-ns#type,rdf:type,3751
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:hasFacility,1640
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:location,1085
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:userRating,1000
4,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nationality,472
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:restaurantType,305
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:inCity,267
8,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:diet,137
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:nextTo,34



--- types / classes (rdf:type) ---
rdf:type triples: 3751
distinct classes: 32


,class_iri,class,instances_count
0,http://www.w3.org/2002/07/owl#NamedIndividual,owl:NamedIndividual,1746
1,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Restaurant,506
2,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hotel,344
3,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Camping_Site,342
4,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Hostel,314
5,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Neighbourhood,267
6,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Museum,55
7,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:City,34
8,http://www.w3.org/2002/07/owl#Class,owl:Class,33
9,http://kai.cs.vu.nl/2024/situated-minor-projec...,ns1:Trainstation,22



--- literals ---
literal objects: 485
predicates with literals: 1
avg literal length: 13.393814432989691
top literal predicates:
    485  rdfs:label
top datatypes:
    485  no-datatype
top languages:


,predicate_iri,predicate,literal_count
0,http://www.w3.org/2000/01/rdf-schema#label,rdfs:label,485


# 2. Create a small ontology that can support expressive inference about hotels and analyse the dialogues (examples.txt).

# Create your own dialogues

Once you have created your ontology, use it as the foundation for designing dialogues. Study the examples in examples.txt to understand their structure and content. Then, create 10 dialogues of your own, ensuring a range of difficulty levels: 5 simple ones and 5 more challenging ones. These dialogues should illustrate how your ontology can support reasoning and should include references to the types of information modeled in your ontology.

In [ ]:
# Create 10 dialoges based on the description
dialogue1: str = ""
dialogue2: str = ""
dialogue3: str = ""
dialogue4: str = ""
dialogue5: str = ""
dialogue6: str = ""
dialogue7: str = ""
dialogue8: str = ""
dialogue9: str = ""
dialogue10: str = ""
dialogues: list = [dialogue1, dialogue2, dialogue3, dialogue4, dialogue5, dialogue6, dialogue7, dialogue8, dialogue9, dialogue10]

# 3. Deploy a reasoning environment

Treated as the ABox in the OWL knowledge base. The idea is that instance queries
with complex class expressions should be used to retrieve different hotels, where
reasoning is crucial for many aspects. For example, a query "give me places that are
close to a coast" would also return places next to a beach if the query is evaluated
together with the ontology that states that a place next to the beach is next to a
coast.

In [22]:
ontology_path: str = dst

In [33]:
from owlready2 import World, sync_reasoner_pellet
from rdflib import Graph, URIRef
from collections import Counter

# load kb and run reasoning
world = World()
kb = world.get_ontology(f"file://{ontology_path}").load(format="ntriples")
print("loaded kb:", kb.base_iri)

with kb:
    sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)
print("reasoning done")

# load data into rdflib for querying
g = Graph()
g.parse(ontology_path, format="nt")

RDF_TYPE = URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

# helper function to get the local name of IRI
def localname(u: URIRef) -> str:
    s = str(u)
    return s.rsplit("#", 1)[-1] if "#" in s else s.rsplit("/", 1)[-1]

# find the class IRI
class_counts = Counter(
    o for s, p, o in g.triples((None, RDF_TYPE, None)) if isinstance(o, URIRef)
)

def pick_class_by_name(name: str):
    for c in class_counts:
        if localname(c).lower() == name.lower():
            return c
    return None

## right now this example code only finds the class "Hotel"
hotel_iri = pick_class_by_name("Hotel")

if hotel_iri is None:
    print("could not find class 'Hotel'. top rdf:type classes are:")
    for cls, c in class_counts.most_common(20):
        print(f"{c:>6}  {cls}")
    raise RuntimeError("Hotel class not found in graph")

print("using hotel class:", hotel_iri)

# query: all "Hotel" instances
q = """
SELECT DISTINCT ?h
WHERE {
  ?h a ?Hotel .
}
"""
rows = list(g.query(q, initBindings={"Hotel": hotel_iri}))

print("\nquery: instances of Hotel")
print("number of hotels:", len(rows))

for (h,) in rows[:50]:
    print(" -", h)

loaded kb: file:///kaggle/working/data.nt#


* Owlready2 * Running Pellet...
    java -Xmx2000M -cp /usr/local/lib/python3.12/dist-packages/owlready2/pellet/httpcore-4.2.2.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jcl-over-slf4j-1.6.4.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jgrapht-jdk1.5.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jena-tdb-0.10.0.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/slf4j-log4j12-1.6.4.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/commons-codec-1.6.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/httpclient-4.2.3.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/log4j-core-2.19.0.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/antlr-runtime-3.2.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/antlr-3.2.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/slf4j-api-1.6.4.jar:/usr/local/lib/python3.12/dist-packages/owlready2/pellet/jena-iri-0.9.5.jar:/usr/local/lib

reasoning done
using hotel class: http://kai.cs.vu.nl/2024/situated-minor-project/hotel#Hotel

query: instances of Hotel
number of hotels: 344
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation312
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation162
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation648
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation259
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation967
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation960
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation146
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation666
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation910
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation455
 - http://kai.cs.vu.nl/2024/situated-minor-project/hotel#accomodation580
 - http://kai.cs.vu.nl/2024/situated-minor-project/hot

# 4. Instruct the LLM to produce the query or components of the query (e.g., keywords) against the KG

In [ ]:
#Download ollama
# For Kaggle or Linux: download with this command, for Windows & Mac locally, download executable from website
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess
process = subprocess.Popen("ollama serve", shell=True) #runs on a different thread

#Download Python library
!pip install ollama

In [ ]:
# Import ollama & pull LLM
import ollama
!ollama pull llama3.2
model: str = "llama3.2"

In [ ]:
#Step 1: Write the instruction for the LLM - remember the overarching topic (assistance with hotels), as well as the fact that
# this step is meant to merely extract queries from the user input.

# Instruct LLM
instruction: str = "..."

In [ ]:
#Step 2: Write a function that takes the model, instruction and one user question as input, runs the LLM and outputs its response
def question_to_query(instruction: str, question: str, model="llama3.2") -> str:
    '''
    This function is meant to use the instruction defined above to run the LLM in order to convert one user input
    question into a query for the ontology reasoner.
    Parameters: instruction (string), question (string), model version (string)
    Returns: LLM response (string)
    '''
    # TODO

In [ ]:
#Step 3: Run the LLM for each example defined above

# Helper function
def find_between(s: str, start: str, end: str) -> str:
    return s.split(start)[1].split(end)[0]

for dialogue in dialogues:
    print("User question:", dialogue)
    print()
    result: str = question_to_query(instruction, dialogue, model)
    # Possibly only extract the relevant parts
    print("Extracted query:", result)
    queries.append(result)
    print()

# 5. Use an LLM to summarize some result into a natural language response to the user.

In [ ]:
#Step 1: Extract knowledge from query with the reasoner and return as list
def reason(query: str) -> list:
    '''
    This function should convert a query into an OWL expression and use the reasoner
    to return the answers.
    '''
    # TODO

In [ ]:
#Step 2: Instruct & run the LLM for the new task: transform the extracted knowledge into a natural language response based
# on the original question
def knowledge_to_response(question: str, knowledge: str, model="llama3.2"):
    '''
    This function is meant to write an instruction based on an item of extracted knowledge and the original user
    question, and run the LLM to summarize a response.
    '''
    # TODO

In [ ]:
#Step 3: Combine everything: generate queries from the dialogues, extract knowledge from queries with the reasoner and
# generate summary responses

# 6. Evaluate your LLM

In [ ]:
# TODO: your code to implement and demonstrate evaluation metrics
# Suggestions: comparison of generated queries with the queries manually created in examples.txt, Intersection Over Union,
# but you can be creative here